# MJO 0: Overview

Example diagrams from [Physical Science Lab](psl.noaa.gov/mjo)
 <p align="center">
    <img src="https://preview.weather.gov/images/arx/lanina/MJO2.jpg">
</p>

 <p align="center">
    <img src="https://psl.noaa.gov/mjo/psl/mjo/images/mjo_global_impacts.jpg">
</p>"

## MJO 1: Obtain Daily Model Data/Output

Read in daily model output via `daily.netcdf.ncl`"

`MJO_suite.py` - Line 44 - "OBTAINING DAILY OUTPUT"
```
generate_ncl_plots(os.environ["POD_HOME"]+"/daily_netcdf.ncl")
```

`daily_netcdf.ncl` reads daily output files from CAM2 and processes the data (each daily file has 30 days of data) (for example: U200(time, lat, lon))

Configuration file: `input_NCEP_Reanalysis_1980_1990.jsonc` (Configuration for MDTF-diagnostics driver script)

Additional details: [see here](https://mdtf-diagnostics.readthedocs.io/en/latest/sphinx/dev_cheatsheet.html#notes)

In [1]:
# Variable-specific environment variables which are accessed with os.environ()
# CASENAME =  ???? (string of file/case name)
# startdate = ???? (string in YYYYMMDD or YYYYMMDDHHMMSS for starting period of analysis)
# enddate =   ???? (string in YYYYMMDD or YYYYMMDDHHMMSS for ending period of analysis)

## DATADIR =  "data/CASENAME/day" (created in MJO_suite if doesn't already exist, L40)
# WORK_DIR =  (working directory) 

# U200_FILE = ???? (data file for 200-hPa zonal wind at lower boundary level, eastward wind in the atmosphere in m/s, scalar-coordinates where atmosphere pressure level lev=200)
# U850_FILE = ???? (data file for 850-hPa zonal wind at higher boundary level, eastward wind in the amotphere in m/s, scalar-coordinates where atmosphere pressure level lev=850)
# V200_FILE = ???? (data file for 200-hPa meridional wind, northward wind in the atmosphere in m/s, scalar-coordaintes where lev=200)
# V850_FILE = ???? (data file for northward wind in m/s, scalar-coordinates where atmospheric pressure level lev = 850)
# RLUT_FILE = ???? (data file for outgoing longwave radiation, aka: OLR)
# PR_FILE =   ???? (data file for percipitation)

# lev_coord =  "lev" (name of the level dimension, e.g. air pressure hPa (postive down, z-axis)
# lat_coord =  "lat" (name of the latitude dimension in the model's native format, e.g. latitude in degrees North, y-axis)
# lon_coord =  "lon" (name of the longitude dimension in the model's native format, e.g. longitude in degrees East, x-axis)
# time_coord = "time" (name of the time dimension in the model's native format)
# pr_var =     "PRECT" (name of precipitation rate, in m/s, with the dimensions = "time, "lat", "lon" -- retained for backwards compatibility -> absolute path to the file containing "pr" data, for example, "/dir/precip.nc")
# rlut_var =   "FLUT" (TOA Outgoing longwave flux/radiation, in W/m^2, with the dimensions = "time", "lat", "lon")

In [2]:
# # getenv == os.environ
import os

CASENAME = "QBOi.EXP1.AMIP.001"
startdate = '19790101'
enddate = '19811231'

# /glade/u/home/bundy/mdtf/MDTF_3_main/MDTF-diagnostics.blocking_notebook/diagnostics/MJO_suite/MJO_driver.py
#DATADIR = "/glade/u/home/bundy/diag/mdtf/inputdata/model/QBOi.EXP1.AMIP.001/"
WORK_DIR = os.getcwd()
DATADIR = "/data/"

U200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U200.day.nc"
V200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V200.day.nc"
U850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U850.day.nc"
V850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V850.day.nc"
RLUT_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.FLUT.day.nc"
PR_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.PRECT.day.nc"

lev_coord = "lev"
lat_coord = "lat"
lon_coord = "lon"
time_coord = "time"
pr_var = "PRECT"
rlut_var = "FLUT"

u200_var = "U200"
v200_var = "V200"

wk_dir = WORK_DIR + "/model/"

# setup data directory, if does not already exist
#if not os.path.exists(DATADIR): os.makedirs(DATADIR)

In [3]:
# check date format
from datetime import datetime

def check_date_format(date_in="", is_start=True):
    # update and check date format

    # if date string is empty
    if date_in == "":
        print(f"ERROR: No date_type given to function check_date_format()")
        return

    # if input is just YYYY, add in MMDDHH suffix
    if len(date_in) == 4:
        # if input if just YYYY, add MMDDHH
        if is_start: # is start date
            date_suffix = str(101 * 100) # 10 10 0 # TODO: month as 1 or 01?
        if not is_start: # is end date
            date_suffix = str(1231 * 100) # 12 31 00

        date_out = date_in + date_suffix
        #print(f"Corrected {date_in} date date to {date_out}")

    # if input is YYYYMMDD, but missing hours
    if len(date_in) == 8:
        date_out = date_in + "00"
        #print(f"Corrected {date_in} date to {date_out}")

    # if input is correct length, check format
    if len(date_in) == 10:
        date_out = date_in

    date_format = "%Y%m%d%H" # YYYYMMDDHH
    try:
        matches = bool(datetime.strptime(date_out, date_format))
    except ValueError:
        matches = False
    #print(f"{date_in} matches {date_format} = {matches}")
    #print(f"Final Date = {date_out}")
    return date_out

startdate = check_date_format(startdate, is_start=True)
enddate = check_date_format(enddate, is_start=False)

In [4]:
# daily_netcdf.ncl
# netCDF variables

print("Starting: Daily NETCDF")
casename = CASENAME
datadir = DATADIR
level = lev_coord
wk_dir = WORK_DIR + "/model/"
if not os.path.exists(wk_dir): os.makedirs(wk_dir)

file_u200 = U200_FILE
file_v200 = V200_FILE
file_u850 = U850_FILE
file_v850 = V850_FILE
file_rlut = RLUT_FILE
file_pr = PR_FILE

print(f"daily_netcdf.ncl reading {file_pr} for time coordinates")
print("Assuming without checking that all have time coordinates")

Starting: Daily NETCDF
daily_netcdf.ncl reading /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.PRECT.day.nc for time coordinates
Assuming without checking that all have time coordinates


In [5]:
import xarray as xr

In [6]:
xr.open_dataset(U200_FILE)

<xarray.Dataset> Size: 565MB
Dimensions:    (time: 2555, nbnd: 2, lat: 192, lon: 288)
Coordinates:
  * time       (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat        (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon        (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: nbnd
Data variables:
    date       (time) int32 10kB ...
    time_bnds  (time, nbnd) object 41kB ...
    U200       (time, lat, lon) float32 565MB ...

In [7]:
xr.open_dataset(V200_FILE)

<xarray.Dataset> Size: 571MB
Dimensions:    (time: 2580, nbnd: 2, lat: 192, lon: 288)
Coordinates:
  * time       (time) object 21kB 1975-01-01 00:00:00 ... 1982-01-25 00:00:00
  * lat        (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon        (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: nbnd
Data variables:
    date       (time) int32 10kB ...
    time_bnds  (time, nbnd) object 41kB ...
    V200       (time, lat, lon) float32 571MB ...
Attributes: (12/17)
    interp_type:               bilinear
    interp_outputgridtype:     equally spaced with poles
    np:                        4
    ne:                        30
    Conventions:               CF-1.0
    source:                    CAM
    ...                        ...
    revision_Id:               $Id$
    initial_file:              /glade/p/work/jrichter/60Lcam5301_B6ORO1F85/ca...
    topography_file:           /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/...
    history:                   Tue Dec 31 16:01:05 2024: ncatted -a calendar,...
    NCO:                       netCDF Operators version 5.1.9 (Homepage = htt...
    nco_openmp_thread_number:  1

In [8]:
xr.open_dataset(U850_FILE)

<xarray.Dataset> Size: 565MB
Dimensions:    (time: 2555, nbnd: 2, lat: 192, lon: 288)
Coordinates:
  * time       (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat        (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon        (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: nbnd
Data variables:
    date       (time) int32 10kB ...
    time_bnds  (time, nbnd) object 41kB ...
    U850       (time, lat, lon) float32 565MB ...

In [9]:
xr.open_dataset(V850_FILE)

<xarray.Dataset> Size: 571MB
Dimensions:  (time: 2580, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 21kB 1975-01-01 00:00:00 ... 1982-01-25 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) int32 10kB ...
    V850     (time, lat, lon) float32 571MB ...
Attributes:
    history:                    Tue Dec 31 16:05:07 2024: ncatted -a calendar...
    history_of_appended_files:  Thu Nov 15 11:43:58 2018: Appended file QBOi....
    NCO:                        netCDF Operators version 5.1.9 (Homepage = ht...

In [10]:
xr.open_dataset(RLUT_FILE)

<xarray.Dataset> Size: 565MB
Dimensions:    (time: 2555, nbnd: 2, lat: 192, lon: 288)
Coordinates:
  * time       (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat        (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon        (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: nbnd
Data variables:
    date       (time) int32 10kB ...
    time_bnds  (time, nbnd) object 41kB ...
    FLUT       (time, lat, lon) float32 565MB ...
Attributes:
    history:  Thu Mar  6 13:31:31 2025: ncatted -a cell_methods,FLUT,m,c,time...
    NCO:      netCDF Operators version 5.3.1 (Homepage = http://nco.sf.net, C...

In [11]:
xr.open_dataset(PR_FILE)

<xarray.Dataset> Size: 565MB
Dimensions:    (time: 2555, nbnd: 2, lat: 192, lon: 288)
Coordinates:
  * time       (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat        (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon        (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: nbnd
Data variables:
    date       (time) int32 10kB ...
    time_bnds  (time, nbnd) object 41kB ...
    PRECT      (time, lat, lon) float32 565MB ...

In [12]:
file_netcdf = xr.open_dataset(file_pr)
print(list(file_netcdf.coords))

['time', 'lat', 'lon']


In [13]:
#print(file_netcdf.variables.keys())
#print("\n")
#print(file_netcdf.variables)

In [14]:
yr1 = int(startdate)
yr2 = int(enddate)
print(yr1)
print(yr2)

1979010100
1981123100


In [15]:
time = file_netcdf.variables[time_coord]
print(time.attrs)
# TODO: find units/calendar, not visible with xarray, but below uses netCDF4
time

{'long_name': 'time', 'bounds': 'time_bnds'}


<xarray.IndexVariable 'time' (time: 2555)> Size: 20kB
array([cftime.DatetimeNoLeap(1975, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 2, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 3, 0, 0, 0, 0, has_year_zero=True), ...,
       cftime.DatetimeNoLeap(1981, 12, 29, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 30, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 31, 0, 0, 0, 0, has_year_zero=True)],
      shape=(2555,), dtype=object)
Attributes:
    long_name:  time
    bounds:     time_bnds

In [16]:
from netCDF4 import Dataset
file_pr_as_dataset = Dataset(file_pr)
time_netcdf = file_pr_as_dataset[time_coord]

In [17]:
print(f"before: \n{time_netcdf.units}\n{time_netcdf.calendar}")

if (time_netcdf.units == "julian day"): # if time units in Julian days, convert
    time_netcdf.units = "days since -4713-01-01 00:00:00"
    time_netcdf.calendar = "julian"

print(f"after: \n{time_netcdf.units}\n{time_netcdf.calendar}")

before: 
days since 1975-01-01 00:00:00
noleap
after: 
days since 1975-01-01 00:00:00
noleap


In [18]:
import numpy as np
days_since_date = time[:] # extract all time days since date
days_since_date

<xarray.IndexVariable 'time' (time: 2555)> Size: 20kB
array([cftime.DatetimeNoLeap(1975, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 2, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 3, 0, 0, 0, 0, has_year_zero=True), ...,
       cftime.DatetimeNoLeap(1981, 12, 29, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 30, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 31, 0, 0, 0, 0, has_year_zero=True)],
      shape=(2555,), dtype=object)
Attributes:
    long_name:  time
    bounds:     time_bnds

In [19]:
# select data for a single year
# file_netcdf.sel(time="1980")

In [20]:
# convert mixed Julian/Gregorian date to a UT-referenced date
# via NCL: cd_calendar --> cd_calendar(time, -3)
# Returns values in YYYYMMDDHH as an integer

# from datetime import datetime, timedelta

# start date as datetime
#start_datetime = datetime(1975, 1, 1, 0, 0, 0)

# collect all datetimes
#time_all = []
#for day_since_start in days_since_date:
#    next_day = (start_datetime + timedelta(days=day_since_start))
#    day_YYYYMMDDHH = next_day.strftime("%Y%m%d%H")
#    time_all.append(int(day_YYYYMMDDHH))
#print(f"Start date = {time_all[0]}, End date = {time_all[-1]}, number of dates = {len(time_all)}")

In [21]:
i1 = 0
i2 = len(time) - 1
print(f"From {i1} to {i2}")
print(f"Time range in file: {time[i1].values} - {time[i2].values}")

From 0 to 2554
Time range in file: 1975-01-01 00:00:00 - 1981-12-31 00:00:00


In [22]:
# confirm date format
start_time = check_date_format(str(yr1), is_start=True)
end_time = check_date_format(str(yr2), is_start=False)

In [23]:
# 24 hours tolerance allows for different time resolutions

#tol = 24

#for i in range(len(time_all)-1):
#    #print(f"examining times {i} {time_all[i]}")
#    if abs(int(time_all[i]) - int(start_time)) < tol:
#        i1 = i
#        print(f"Found start_time {time_all[i]} {end_time}")
#    if abs(int(time_all[i]) - int(end_time)) < tol:    
#        i2 = i
#        print(f"Found end_time {time_all[i]} {end_time}")

#print(f"Time range indices: {i1} {time_all[i1]} - {i2} {time_all[i2]}")

In [24]:
filtered_file = time[i1:i2]
print(f"Using date range: {min(filtered_file).values} to {max(filtered_file).values}")
ndays = len(filtered_file)
print(f"Number of days: {ndays}")

Using date range: 1975-01-01 00:00:00 to 1981-12-30 00:00:00
Number of days: 2554


In [25]:
# check if data array is monotonic (strictly increasing)
import pandas as pd
def check_monotonic(array):
    # Check for increasing order sequence
    is_increasing = pd.Index(array).is_monotonic_increasing
    try:
        assert is_increasing
        return is_increasing
    except Exception:
        print("ERROR: daily_netcdf.ncl finds dates are not monotonic increasing")
    
print(f"Is monotonic = {check_monotonic(filtered_file)}")

Is monotonic = True


In [26]:
# First just use read_model_file for pr here, then put into var loop below
from netCDF4 import Dataset

def read_dim(f, var_name, opt=None, upper_bound=None, lower_bound=None):
    print("running read_dim")
    # filter lat bounds from 40 to -40
    routine_name = "read_dim"
    verbose = False
    if opt:
        if lower_bound is not None:
            #print(f"found lower_bound {lower_bound}")
            pass
        if upper_bound is not None:
            #print(f"found upper_bound {upper_bound}")
            pass
    
    if lower_bound is not None and upper_bound is not None:
        # filter based on upper and lower bound
        var = f[var_name].where((f[var_name] > lower_bound) & (f[var_name] < upper_bound))
    else:
        var = f[var_name]

    return var    

def get_gw(f, latS, latN):
    # gaussian weight for the latitude dimension
    if False:
    #if "gw" in file_netcdf.coords:
        # remove any existing "gw" dimension
        #f.drops_vars("gw")
        #new_gw = f[gw[lat_coord|latS:latN]]
        print("TODO")
    else:
        print("running get_gw")
        lat = read_dim(f, lat_coord, opt=False, upper_bound=latN, lower_bound=latS) # cut full range based on upper and lower bounds
        nlat = lat.count().item()
        slat = xr.DataArray(coords=(range(nlat+1), ))
        #gw = xr.DataArray(coords=(range(nlat+1), ), dims="gw")
        new_gw = {}

        if lat[0].values < lat[1].values:
            slat[0] = -90.0
            slat[nlat] = 90.0
        else:
            slat[0] = 90.0
            slat[nlat] = -90.0

        for i in range(nlat-1):
            slat[i] = (lat[i-1].values + lat[i].values)/2

        for i in range(nlat-1):
            new_gw[i] = abs(np.sin(slat[i+1] / (180*np.pi)) - np.sin(slat[i] / (180 * np.pi)))
        
    return new_gw

def pinterp(var_name=None, plev=None, fin=None, fin_ps=None):
    print(f"pinterop() interpolating {var_name} to pressure level {plev} mbar")        

def read_model_file(var_name_in=None, file_in=None,
                   var_name_out=None, file_out=None,
                   delete_existing=False,
                   i1=None, i2=None, time_coord=None,
                   lat_coord=None, lon_coord=None,
                   date=None, interp_opts=None,
                   var_name_in_3d=None, plev=None, file_3d=None, file_ps=None):
    # utils.ncl - L186
    # re-write model file
    # writing a file with only necessary time dimension and lat bounds and adding in gw
    # interp_opts = optional args for interpolating in space
    # var_name_in_3d, plev, file_3d, file_ps = optional args for pressure interp
    print("running read_model_file")

    latS = -40
    latN = 40
    opt = True
    lower_bound = latS
    upper_bound = latN

    if os.path.exists(file_out) and not delete_existing: # do not write new file
        # TODO: FIX
        print("file output exists and delete_existing is False")
        print(f"WARNING: using existing file {file_out}")
        print(f"To override, change delete_existing to True")
        if interp_opts is not None:
            #f_out_for_interp = Dataset(file_out)
            f_out_for_interp = xr.open_dataset(file_out)
            var = f_out_for_interp[var_name_out]
            gw = get_gw(f_out_for_interp, latS, latN)
            interp_save_res(var_name_out, interp_opts, var["lat"], var["lon"], gw)
            del(f_out_for_interp)
    else:
        print("write new file")
        if os.path.exists(file_out):
            print(f"WARNING: over-writing existing file, removing existing {file_out}")
            os.remove(file_out)
        if os.path.exists(file_in):
            # CURRENTLY RUNNING:
            print(f"opening {var_name_in} from {file_in}")
            f_in = xr.open_dataset(file_in)
            lat = read_dim(f_in, lat_coord, opt, upper_bound=upper_bound, lower_bound=lower_bound)
            opt = False
            lon = read_dim(f_in, lon_coord, opt, upper_bound=upper_bound, lower_bound=lower_bound)
            gw = get_gw(f_in, latS, latN)

            print("FILE INPUT (while writing new file):")
            print(var_name_in)
            time_start = f_in[var_name_in][i1].time.values
            time_end = f_in[var_name_in][i2].time.values
            print(time_start, time_end)
            ds = f_in[var_name_in]
            #ds = ds.expand_dims({"gw": gw}) -> makes file too large (21 GB)
            var = ds.sel(time=slice(time_start, time_end), lat=slice(-40, 40)) # L245 (utils.ncl)
            print(var)
        else:
            print(f"WARNING: file does not exist for MJO diagnostics: {file_in}")
            print(f"Looking for file matching {var_name_in_3d} at plev {plev}")
            
            # needs checks for var_name_in_3d, plev and file_ps
            if file_ps is None:
                print(f"ERROR: can't find required PS file: {file_ps}")
                return

            f_3d = xr.open_dataset(file_3d)
            f_ps = xr.open_dataset(file_ps)
            # TODO: fill out pinterp() function
            var_1 = pinterp(var_namae_in_3d, plev, f_3d, f_ps) #TODO
            var = ds.sel(time=slice(time_start, time_end), lat=slice(-40, 40)) # L245 (utils.ncl)
            
            if interp_opts:
                print("INTERP_OPTS IS TRUE")
                # TODO

    # Writing new file
    print(f"Creating output file {file_out}")
    try:
        file_out.close()
    except:
        pass
    
    print(f"WRITING OUT NEW NETCDF FILE: {file_out}")
    file_output_xarray = var[time_coord]
    file_output_xarray = file_output_xarray.expand_dims({"gw": gw}) # temp solution: write to a new file
    print(f"file_output_xarray = {file_output_xarray}")
    file_output_xarray.to_netcdf(file_out)

In [27]:
# precipitation rate (pr)
print(f"daily_netcdf.ncl reading {file_pr} for making precip file!\n")
delete_existing = True # True = overwrite, False = don't overwrite

if os.path.exists(file_pr):
    print(f"found pr input file = {file_pr}")
    var_name = pr_var
    var_name_out = "pr"
    file_in = file_pr
    file_out = wk_dir + CASENAME + "." + var_name_out + ".day.nc"
    print(f"file_out = {file_out}\n")
    var_name_3d_model = "not provided"
    file_in_3d = "not provided"
    file_in_ps = "not provided"
    plev = 0

    interp_opts = True # store this field as base resolution no matter what
    interp_to_var_name = var_name_out
    read_model_file(var_name, file_in, var_name_out, file_out, 
                    delete_existing, i1, i2, 
                    time_coord, lat_coord, lon_coord, filtered_file, interp_opts,
                    var_name_3d_model, plev, file_in_3d, file_in_ps)
else:
    print("ERROR: daily pr input files does not exist for MJO diagnostics")

daily_netcdf.ncl reading /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.PRECT.day.nc for making precip file!

found pr input file = /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.PRECT.day.nc
file_out = /home/cs/Github/geocat-research/mjo/model/QBOi.EXP1.AMIP.001.pr.day.nc

running read_model_file
write new file
opening PRECT from /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.PRECT.day.nc
running read_dim
running read_dim
running get_gw
running read_dim
FILE INPUT (while writing new file):
PRECT
1975-01-01 00:00:00 1981-12-31 00:00:00
<xarray.DataArray 'PRECT' (time: 2555, lat: 84, lon: 288)> Size: 247MB
[61810560 values with dtype=float32]
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 672B -39.11 -38.17 -37.23 ... 37.23 38.17 39.11
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Attributes:
    Sampling_Sequence:  rad_lwsw
    units:           

In [28]:
plevs = [850, 200]
var_names = ["u", "v"]

file_u200 = U200_FILE
file_v200 = V200_FILE
file_u850 = U850_FILE
file_rlut = RLUT_FILE
file_pr = PR_FILE

for i in range(len(plevs)):
    plev = plevs[i]
    for j in range(len(var_names)):
        var_name = var_names[j].upper() + str(plev)
        var_name_3d_model = var_names[j] + "_var" # as read from history files (3d field)
        var_name_plev_model = var_names[j] + str(plev) + "_var" # as read from history files (pressure slice)
        var_name_plev_package = var_names[j] + str(plev) # new file name and varname in file
        file_var_name = var_names[j].upper() + str(plev) + "_FILE"
        if file_var_name == "U850_FILE":
            file_in = file_u850
        if file_var_name == "V850_FILE":
            file_in = file_v850
        if file_var_name == "U200_FILE":
            file_in = file_u200
        if file_var_name == "V200_FILE":
            file_in = file_v200
        print(f"file_in = {file_in}")
        file_out = wk_dir + CASENAME + "." + var_name_plev_model + ".day.nc"
        print(f"\tfile_out   = {file_out}")
        # all files supplied to POD are 3D slices -> PS not part of varlist request
        file_in_3d = WORK_DIR + DATADIR + CASENAME + "." + var_name_3d_model + ".day.nc"
        #print(f"\tfile_in_3d = {file_in_3d}")
        file_in_ps = WORK_DIR + DATADIR + CASENAME + ".PS.day.nc"
        #print(f"\tfile_in_ps = {file_in_ps}")
        
        read_model_file(var_name, file_in, var_name_out, file_out, 
                    delete_existing, i1, i2, 
                    time_coord, lat_coord, lon_coord, filtered_file, interp_opts,
                    var_name_3d_model, plev, file_in_3d, file_in_ps)

file_in = /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.U850.day.nc
	file_out   = /home/cs/Github/geocat-research/mjo/model/QBOi.EXP1.AMIP.001.u850_var.day.nc
running read_model_file
write new file
opening U850 from /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.U850.day.nc
running read_dim
running read_dim
running get_gw
running read_dim
FILE INPUT (while writing new file):
U850
1975-01-01 00:00:00 1981-12-31 00:00:00
<xarray.DataArray 'U850' (time: 2555, lat: 84, lon: 288)> Size: 247MB
[61810560 values with dtype=float32]
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 672B -39.11 -38.17 -37.23 ... 37.23 38.17 39.11
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Attributes:
    Sampling_Sequence:  rad_lwsw
    units:              m/s
    long_name:          Zonal wind at 850 mbar pressure surface
    cell_methods:       time: mean                           

In [29]:
# outgoing longwave radiation (rlut)

if os.path.exists(file_rlut):
    var_name_out = "rlut"
    var_name = rlut_var
    file_in = file_rlut
    print(f"file_in = {file_in}")
    file_out = wk_dir + CASENAME + "." + var_name_out + ".day.nc"
    print(f"\tfile_out   = {file_out}")
    var_name_3d_model = "not provided"
    file_in_3d = "not provided"
    file_in_ps = "not provided"
    plev = 0

    read_model_file(var_name, file_in, var_name_out, file_out, 
                    delete_existing, i1, i2, 
                    time_coord, lat_coord, lon_coord, filtered_file, interp_opts,
                    var_name_3d_model, plev, file_in_3d, file_in_ps)
else:
    print("daily rlut file does not exist for MJO diagnostics")

file_in = /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.FLUT.day.nc
	file_out   = /home/cs/Github/geocat-research/mjo/model/QBOi.EXP1.AMIP.001.rlut.day.nc
running read_model_file
write new file
opening FLUT from /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.FLUT.day.nc
running read_dim
running read_dim
running get_gw
running read_dim
FILE INPUT (while writing new file):
FLUT
1975-01-01 00:00:00 1981-12-31 00:00:00
<xarray.DataArray 'FLUT' (time: 2555, lat: 84, lon: 288)> Size: 247MB
[61810560 values with dtype=float32]
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 672B -39.11 -38.17 -37.23 ... 37.23 38.17 39.11
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Attributes:
    cell_methods:       time: mean
    long_name:          Upwelling longwave flux at top of model
    units:              W/m2
    Sampling_Sequence:  rad_lwsw
Creating output file /home/cs